# BugBuster HAT — RP2040 Firmware Build & Install

This notebook builds the RP2040 firmware from source and flashes it to the device.  
It is **system- and directory-agnostic**: it auto-detects the firmware root, the host OS, and available tools.

**Supported hosts:** macOS (Apple Silicon & Intel), Linux (x86-64 & aarch64)  
**Flash methods (in order of preference):** picotool → UF2 drag-drop  

---
## How to use
1. Run **Cell 1** first — it sets up paths and helpers used by all subsequent cells.
2. Run **Cell 2** to check / install dependencies.
3. Run **Cell 3** to initialise git submodules (only needed once).
4. Run **Cell 4** to configure the CMake build.
5. Run **Cell 5** to compile the firmware.
6. Run **Cell 6** to flash the firmware onto the RP2040.

> **Tip:** _Kernel → Restart & Run All_ performs a full clean build + flash in one shot.

In [ ]:
# ── Cell 1 · Setup: paths, OS detection, helper utilities ────────────────────
import os, sys, subprocess, shutil, platform, textwrap, time
from pathlib import Path

# ── Locate firmware root ─────────────────────────────────────────────────────
# Searches for the RP2040 CMakeLists.txt that declares the bugbuster_hat target.
# Works regardless of where this notebook lives in the repo tree:
#   • Inside the firmware directory  (e.g. Firmware/RP2040/notebooks/)
#   • As a sibling of Firmware/      (e.g. notebooks/ at repo root)
#   • Any other location             (shallow glob fallback)
# Override at any time by setting the FIRMWARE_ROOT environment variable.

def _find_firmware_root() -> Path:
    MARKER = "bugbuster_hat"

    def _is_firmware_root(p: Path) -> bool:
        cmake = p / "CMakeLists.txt"
        return cmake.exists() and MARKER in cmake.read_text(errors="ignore")

    # Strategy 0: explicit env override
    override = os.environ.get("FIRMWARE_ROOT")
    if override:
        p = Path(override).expanduser().resolve()
        if _is_firmware_root(p):
            return p
        raise FileNotFoundError(
            f"FIRMWARE_ROOT={override!r} has no CMakeLists.txt with '{MARKER}'"
        )

    # Build a de-duplicated list of ancestor directories, starting from the
    # notebook's own directory (works in VS Code, JupyterLab, classic Jupyter)
    # and falling back to cwd.
    ancestors: list[Path] = []
    try:
        nb_dir = Path(globals().get("__vsc_ipynb_file__") or __file__).parent.resolve()
        for p in [nb_dir] + list(nb_dir.parents):
            if p not in ancestors:
                ancestors.append(p)
    except NameError:
        pass
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if p not in ancestors:
            ancestors.append(p)

    # Strategy 1: direct match on each ancestor — works when the notebook is
    # inside the firmware tree (e.g. Firmware/RP2040/notebooks/).
    for candidate in ancestors:
        if _is_firmware_root(candidate):
            return candidate

    # Strategy 2: well-known sibling paths — covers the common layout where
    # notebooks/ and Firmware/ are siblings at the repo root.
    KNOWN_SUBPATHS = [
        "Firmware/RP2040", "firmware/rp2040",
        "Firmware/rp2040", "firmware/RP2040",
        "RP2040", "rp2040",
    ]
    for root in ancestors:
        for sub in KNOWN_SUBPATHS:
            candidate = root / sub
            if _is_firmware_root(candidate):
                return candidate

    # Strategy 3: shallow glob — 1 and 2 levels down from the first few
    # ancestors (avoids scanning the whole filesystem).
    for root in ancestors[:4]:
        for cmake in sorted(root.glob("*/CMakeLists.txt")) + sorted(root.glob("*/*/CMakeLists.txt")):
            if MARKER in cmake.read_text(errors="ignore"):
                return cmake.parent

    raise FileNotFoundError(
        "Could not locate the RP2040 firmware root (CMakeLists.txt with "
        f"'{MARKER}').\nSet the FIRMWARE_ROOT environment variable to the "
        "correct directory and re-run."
    )

FIRMWARE_ROOT = _find_firmware_root()
BUILD_DIR     = FIRMWARE_ROOT / "build"
UF2_PATH      = BUILD_DIR / "bugbuster_hat.uf2"

# ── OS / architecture detection ──────────────────────────────────────────────
HOST_OS   = platform.system()          # 'Darwin' | 'Linux' | 'Windows'
HOST_ARCH = platform.machine().lower() # 'arm64' | 'x86_64' | 'aarch64'
IS_MAC    = HOST_OS == "Darwin"
IS_LINUX  = HOST_OS == "Linux"
IS_WIN    = HOST_OS == "Windows"

# ── Utilities ────────────────────────────────────────────────────────────────
def run(cmd, *, cwd=None, check=True, capture=False):
    """Run a shell command, streaming output in real time (or capturing it)."""
    cwd = cwd or FIRMWARE_ROOT
    if isinstance(cmd, str):
        cmd = cmd.split()
    if capture:
        result = subprocess.run(cmd, cwd=cwd, check=check,
                                capture_output=True, text=True)
        return result.stdout.strip()
    result = subprocess.run(cmd, cwd=cwd, check=check)
    return result.returncode == 0

def which(name):
    return shutil.which(name)

def ok(msg):   print(f"  ✓  {msg}")
def warn(msg): print(f"  ⚠  {msg}")
def err(msg):  print(f"  ✗  {msg}")
def hdr(msg):  print(f"\n{'─'*60}\n  {msg}\n{'─'*60}")

hdr("Environment")
print(f"  Firmware root : {FIRMWARE_ROOT}")
print(f"  Build dir     : {BUILD_DIR}")
print(f"  Host OS       : {HOST_OS} ({HOST_ARCH})")
print(f"  Python        : {sys.version.split()[0]}")

In [ ]:
# ── Cell 2 · Dependency check ────────────────────────────────────────────────
import re, glob as _glob

hdr("Dependency Check")

missing_required = []
missing_optional = []

def _ver(cmd):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True)
        m = re.search(r'[\d]+\.[\d]+(?:\.[\d]+)?', r.stdout + r.stderr)
        return m.group(0) if m else "(unknown version)"
    except Exception:
        return None

# ── cmake, make, git, python3 ────────────────────────────────────────────────
for tool in ["cmake", "make", "git", "python3"]:
    p = which(tool)
    if p:
        ok(f"{tool} {_ver([tool, '--version'])}  →  {p}")
    else:
        err(f"{tool} not found")
        missing_required.append(tool)

# ── arm-none-eabi toolchain ───────────────────────────────────────────────────
# Some macOS packages ship gcc without a newlib sysroot (e.g. the Homebrew
# arm-none-eabi-gcc 15.x formula).  We pick the first candidate that can
# compile a minimal ARM C file that includes <stdint.h>.

def _has_sysroot(gcc: str) -> bool:
    """Return True if this gcc has a working newlib sysroot (stdint.h reachable)."""
    try:
        r = subprocess.run(
            [gcc, "-mcpu=cortex-m0plus", "-mthumb",
             "-x", "c", "-", "-c", "-o", "/dev/null"],
            input="#include <stdint.h>\nint x = 0;\n",
            capture_output=True, text=True, timeout=10,
        )
        return r.returncode == 0
    except Exception:
        return False

def _arm_candidates() -> list[str]:
    seen, out = set(), []
    def _add(p):
        rp = str(Path(p).resolve())
        if Path(rp).exists() and rp not in seen:
            seen.add(rp); out.append(rp)
    w = which("arm-none-eabi-gcc")
    if w: _add(w)
    for pat in [
        "/opt/homebrew/share/arm-gnu-toolchain-*/bin/arm-none-eabi-gcc",
        "/usr/local/share/arm-gnu-toolchain-*/bin/arm-none-eabi-gcc",
        "/Applications/ArmGNUToolchain/*/arm-none-eabi/bin/arm-none-eabi-gcc",
    ]:
        for p in sorted(_glob.glob(pat), reverse=True):
            _add(p)
    return out

ARM_GCC = None   # consumed by Cell 4
ARM_GXX = None

print("  Probing arm-none-eabi-gcc candidates for working sysroot …")
for candidate in _arm_candidates():
    ver = _ver([candidate, "--version"]) or "?"
    gxx = str(Path(candidate).parent / "arm-none-eabi-g++")
    if not _has_sysroot(candidate):
        warn(f"arm-none-eabi-gcc {ver}  →  {candidate}  (no newlib sysroot — skip)")
        continue
    if not Path(gxx).exists():
        warn(f"arm-none-eabi-gcc {ver}  →  {candidate}  (no g++ sibling — skip)")
        continue
    ok(f"arm-none-eabi-gcc {ver}  →  {candidate}")
    ok(f"arm-none-eabi-g++         →  {gxx}")
    ARM_GCC, ARM_GXX = candidate, gxx
    break

if ARM_GCC is None:
    err("No arm-none-eabi-gcc with complete newlib sysroot found")
    if IS_MAC:
        print("  brew install arm-gnu-toolchain   # 13.x — includes full sysroot")
    else:
        print("  sudo apt install gcc-arm-none-eabi binutils-arm-none-eabi")
    missing_required.append("arm-none-eabi-gcc")

# ── picotool (optional) ───────────────────────────────────────────────────────
local_picotool = BUILD_DIR / "picotool" / "picotool"
picotool_path = which("picotool") or (str(local_picotool) if local_picotool.exists() else None)
if picotool_path:
    ok(f"picotool  →  {picotool_path}")
else:
    warn("picotool not found — will use UF2 drag-drop for flashing")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
if missing_required:
    raise SystemExit("Resolve the issues above, then re-run this cell.")
ok("All required dependencies found.")

In [ ]:
# ── Cell 3 · Initialise git submodules ───────────────────────────────────────
#
# The Pico SDK and debugprobe live as git submodules inside lib/.
# This cell is idempotent — safe to re-run.

hdr("Git Submodules")

pico_sdk   = FIRMWARE_ROOT / "lib" / "pico-sdk"
debugprobe = FIRMWARE_ROOT / "lib" / "debugprobe"

def _submodule_ready(path: Path) -> bool:
    """True if the submodule directory is non-empty."""
    return path.is_dir() and any(path.iterdir())

if _submodule_ready(pico_sdk) and _submodule_ready(debugprobe):
    ok("Submodules already initialised — skipping")
else:
    print("  Initialising submodules (this may take a while on first run) …")
    run(["git", "submodule", "update", "--init", "--recursive"],
        cwd=FIRMWARE_ROOT)
    ok("Submodules ready")

# Verify key paths exist
for p, label in [
    (pico_sdk   / "CMakeLists.txt", "pico-sdk"),
    (debugprobe / "CMakeLists.txt", "debugprobe"),
    (debugprobe / "freertos",       "FreeRTOS kernel"),
]:
    if p.exists():
        ok(f"{label}  →  {p.parent}")
    else:
        err(f"{label} not found at {p}")
        raise FileNotFoundError(f"Missing submodule: {p}")

In [ ]:
# ── Cell 4 · CMake configure ──────────────────────────────────────────────────
#
# Uses ARM_GCC/ARM_GXX selected by Cell 2 (the toolchain with a complete
# newlib sysroot).  If the cached compiler differs, the cmake cache is cleared
# so cmake re-detects with the correct toolchain.
#
# After cmake runs it updates boot-stage-2 link.txt timestamps, which would
# cause make to attempt a relink that fails on some toolchains.  We touch the
# existing bs2 artifacts immediately after cmake so make skips that step.
#
# To force a full clean rebuild (including boot-stage-2):
#   import shutil; shutil.rmtree(BUILD_DIR, ignore_errors=True)

hdr("CMake Configure")

BUILD_DIR.mkdir(exist_ok=True)
cmake_cache = BUILD_DIR / "CMakeCache.txt"
BOARDS_DIR  = FIRMWARE_ROOT / "boards"

# Clear cache if the compiler is changing so cmake re-initialises cleanly.
if cmake_cache.exists():
    cached_cc = ""
    for line in cmake_cache.read_text().splitlines():
        if line.startswith("CMAKE_C_COMPILER:"):
            cached_cc = line.split("=", 1)[-1].strip()
            break
    if cached_cc and Path(cached_cc).resolve() != Path(ARM_GCC).resolve():
        warn(f"Switching compiler: {cached_cc} → {ARM_GCC}")
        warn("Clearing cmake cache.")
        cmake_cache.unlink()

cmake_args = [
    "cmake",
    f"-DCMAKE_C_COMPILER={ARM_GCC}",
    f"-DCMAKE_CXX_COMPILER={ARM_GXX}",
    f"-DCMAKE_ASM_COMPILER={ARM_GCC}",
    f"-DPICO_SDK_PATH={FIRMWARE_ROOT / 'lib' / 'pico-sdk'}",
    f"-DFREERTOS_KERNEL_PATH={FIRMWARE_ROOT / 'lib' / 'debugprobe' / 'freertos'}",
    "-DPICO_BOARD=bugbuster_hat",
    f"-DPICO_BOARD_HEADER_DIRS={BOARDS_DIR}",
    "-DCMAKE_BUILD_TYPE=RelWithDebInfo",
    "..",
]

print(f"  Working dir : {BUILD_DIR}")
print(f"  Compiler    : {ARM_GCC}")
print()

run(cmake_args, cwd=BUILD_DIR)

# cmake updates link.txt on every reconfigure, making make think bs2_default
# needs relinking.  Touch all existing bs2 artifacts so make sees them as
# up-to-date regardless of which toolchain the pico-sdk uses for that step.
BS2_BUILD = BUILD_DIR / "pico-sdk" / "src" / "rp2040" / "boot_stage2"
_preserved = []
for target in [
    BS2_BUILD / "bs2_default.elf",
    BS2_BUILD / "bs2_default.bin",
    BS2_BUILD / "CMakeFiles" / "bs2_default.dir" / "compile_time_choice.S.o",
    BS2_BUILD / "bs2_default_padded_checksummed.S",
    BS2_BUILD / "CMakeFiles" / "bs2_default_library.dir" / "bs2_default_padded_checksummed.S.o",
]:
    if target.exists():
        target.touch()
        _preserved.append(target.name)

ok("CMake configuration complete")
if _preserved:
    ok(f"Preserved boot-stage-2 artifacts: {', '.join(_preserved)}")

In [ ]:
# ── Cell 5 · Build firmware ───────────────────────────────────────────────────
#
# Runs make with all available CPU cores.
# Output: build/bugbuster_hat.uf2  (and .elf / .bin / .hex)

import multiprocessing

hdr("Build Firmware")

nproc = multiprocessing.cpu_count()
print(f"  Building with {nproc} parallel jobs …")
print()

t0 = time.time()
try:
    run(["make", f"-j{nproc}"], cwd=BUILD_DIR)
except subprocess.CalledProcessError as e:
    print()
    err(f"Build failed (exit code {e.returncode}) — see make output above.")
    raise SystemExit(1) from None

elapsed = time.time() - t0

print()
if UF2_PATH.exists():
    uf2_size_kb = UF2_PATH.stat().st_size / 1024
    ok(f"Build succeeded in {elapsed:.1f}s")
    ok(f"UF2 image : {UF2_PATH}  ({uf2_size_kb:.0f} KB)")
else:
    err(f"Build finished but {UF2_PATH} not found — check output above")
    raise FileNotFoundError(UF2_PATH)

In [ ]:
# ── Cell 6 · Flash firmware ───────────────────────────────────────────────────
#
# Strategy A (automatic): picotool — works while RP2040 is in BOOTSEL mode OR
#                          while the firmware is running (picotool reboot -f first).
#
# Strategy B (manual)   : UF2 drag-drop — copy bugbuster_hat.uf2 to the RPI-RP2
#                          USB mass-storage volume that appears when you hold
#                          BOOTSEL while plugging in USB.
#
# Set FLASH_METHOD = 'picotool' | 'dragdrop' | 'auto'  to override detection.

FLASH_METHOD = "auto"   # ← change here if needed

hdr("Flash Firmware")

if not UF2_PATH.exists():
    err(f"UF2 not found at {UF2_PATH}")
    raise FileNotFoundError("Run Cell 5 first to build the firmware.")

# Re-check picotool (may have been installed after Cell 2)
picotool_exe = which("picotool")
if not picotool_exe:
    lp = BUILD_DIR / "picotool" / "picotool"
    if lp.exists():
        picotool_exe = str(lp)

def _rpi_rp2_mount() -> Path | None:
    """Return the mount-point of the RPI-RP2 USB drive, or None."""
    candidates = []
    if IS_MAC:
        candidates = list(Path("/Volumes").glob("RPI-RP2*"))
    elif IS_LINUX:
        import getpass
        user = getpass.getuser()
        candidates  = list(Path("/media").glob(f"{user}/RPI-RP2*"))
        candidates += list(Path("/media").glob("RPI-RP2*"))
        candidates += list(Path("/mnt").glob("RPI-RP2*"))
        candidates += list(Path("/run/media").glob(f"{user}/RPI-RP2*"))
    return candidates[0] if candidates else None

def _try_picotool_flash() -> bool:
    """Attempt to flash via picotool. Returns True on success."""
    print("  Trying picotool …")
    # First try: device already in BOOTSEL mode
    r = subprocess.run(
        [picotool_exe, "load", "-x", str(UF2_PATH)],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(r.stdout)
        return True
    # Second try: reboot into BOOTSEL if firmware is already running
    print("  Device not in BOOTSEL mode — attempting picotool reboot …")
    reboot = subprocess.run(
        [picotool_exe, "reboot", "-f", "-u"],
        capture_output=True, text=True
    )
    if reboot.returncode == 0:
        print("  Rebooting into BOOTSEL … waiting 3 s")
        time.sleep(3)
        r2 = subprocess.run(
            [picotool_exe, "load", "-x", str(UF2_PATH)],
            capture_output=True, text=True
        )
        if r2.returncode == 0:
            print(r2.stdout)
            return True
    print("  picotool could not reach the device.")
    print(f"  stderr: {r.stderr.strip()}")
    return False

def _try_dragdrop_flash() -> bool:
    """Attempt UF2 drag-drop copy. Returns True on success."""
    mount = _rpi_rp2_mount()
    if mount:
        dest = mount / UF2_PATH.name
        print(f"  RPI-RP2 volume found at {mount}")
        print(f"  Copying {UF2_PATH.name} …")
        shutil.copy2(str(UF2_PATH), str(dest))
        ok("UF2 copied — RP2040 will reset and start the new firmware")
        return True
    return False

# ── Choose and execute flash method ───────────────────────────────────────────
flashed = False

if FLASH_METHOD in ("auto", "picotool") and picotool_exe:
    flashed = _try_picotool_flash()
    if flashed:
        ok(f"Flashed via picotool → {UF2_PATH.name}")

if not flashed and FLASH_METHOD in ("auto", "dragdrop"):
    flashed = _try_dragdrop_flash()

if not flashed:
    print()
    warn("Automatic flashing failed or no device detected.")
    print()
    print("  ── Manual UF2 drag-drop instructions ──────────────────────────")
    print("  1. Hold the BOOTSEL button on the RP2040.")
    print("  2. While holding BOOTSEL, connect the RP2040 to your computer via USB.")
    print("  3. A USB drive called RPI-RP2 will appear.")
    print(f"  4. Copy the UF2 file to that drive:")
    print(f"       {UF2_PATH}")
    print("  5. The RP2040 will reboot automatically and run the new firmware.")
    print()
    print("  ── Or re-run this cell after connecting the device ─────────────")